# Binary Classification with a Bank Churn Dataset
(Playground Series - Season 6, Episode 3)

https://www.kaggle.com/competitions/playground-series-s6e3/data

![Image](./data/picture.png)

__About the Tabular Playground Series__
The goal of the Tabular Playground Series is to provide the Kaggle community with a variety of fairly light-weight challenges that can be used to learn and sharpen skills in different aspects of machine learning and data science. The duration of each competition will generally only last a few weeks, and may have longer or shorter durations depending on the challenge. The challenges will generally use fairly light-weight datasets that are synthetically generated from real-world data, and will provide an opportunity to quickly iterate through various model and feature engineering ideas, create visualizations, etc.

__Synthetically-Generated Datasets__
Using synthetic data for Playground competitions allows us to strike a balance between having real-world data (with named features) and ensuring test labels are not publicly available. This allows us to host competitions with more interesting datasets than in the past. While there are still challenges with synthetic data generation, the state-of-the-art is much better now than when we started the Tabular Playground Series two years ago, and that goal is to produce datasets that have far fewer artifacts. Please feel free to give us feedback on the datasets for the different competitions so that we can continue to improve!

__Dataset Description__
The dataset for this competition (both train and test) was generated from a deep learning model trained on the customer churn prediction dataset. Feature distributions are close to, but not exactly the same, as the original. Feel free to use the original dataset as part of this competition, both to explore differences as well as to see whether incorporating the original in training improves model performance.

Files
* train.csv - the training dataset; Exited is the binary target
* test.csv - the test dataset; your objective is to predict the probability of Exited
* sample_submission.csv - a sample submission file in the correct format

Models

1. K-Nearest Neighboor Model            x
2. Gaussian Naive Bayes Model           x
3. Logistic Regressor                   x
4. Support Vector Classification Model  x
5. Decision Tree Model                  x
6. Random Forest Model                  x
7. Linear Discriminant Analysis Model   x
8. Gradient Boosting Classifier Model   x
9. Neural Network CLassifier Model      x

In [1]:
# Installing all packges
%pip install -r ./requirements.txt

  Using cached imblearn-0.0-py2.py3-none-any.whl.metadata (355 bytes)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import all necessary libraries

import os
import time
import numpy as np
import pandas as pd
import pickle as pkl
import seaborn as sns
import datetime as dt
import warnings as wn
import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

In [3]:
# Ignore all warnings
wn.filterwarnings('ignore')

In [4]:
# Set all variables paths 

_plots = './plots/'
_test = './data/test.csv'
_train = './data/train.csv'
_model = './model/model.pkl'
_info = './model/model.docx'
_submission = './data/submission.csv'


In [5]:
# Simplify the process of integrating large CSV

def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.        
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

def import_data(file, **kwargs):
    """create a dataframe and optimize its memory usage"""
    df = pd.read_csv(file, parse_dates=True, **kwargs)
    df = reduce_mem_usage(df)
    return df

In [6]:
# Read the training and testing datasets and preprocess
train = import_data(_train, index_col = "id")
test = import_data(_test, index_col = 'id')

Memory usage of dataframe is 95.20 MB
Memory usage after optimization is: 17.00 MB
Decreased by 82.1%
Memory usage of dataframe is 38.86 MB
Memory usage after optimization is: 7.04 MB
Decreased by 81.9%


In [8]:
# Display the first n rows of training
train.head(n=10)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
id,,,,,,,,,,,,,,,,,,,,
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.093750,1654.000000,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.500000,3778.000000,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.375000,5840.000000,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.687500,70.687500,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.437500,70.437500,Yes
5,Male,0,Yes,Yes,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),20.203125,20.203125,No
6,Female,0,Yes,Yes,24,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.406250,533.500000,No
7,Male,0,Yes,No,72,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,No,Electronic check,92.000000,6828.000000,No
8,Female,1,No,No,1,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,79.562500,79.562500,Yes


In [9]:
# Describe the data
train.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.00000,594194.000000
mean,0.114102,36.577258,NaN,NaN
std,0.317936,25.061922,0.00000,NaN
min,0.000000,1.000000,18.25000,18.796875
25%,0.000000,12.000000,29.90625,639.500000
50%,0.000000,35.000000,74.12500,1434.000000
75%,0.000000,62.000000,90.81250,4264.000000
max,1.000000,72.000000,118.75000,8688.000000


In [10]:
# Display more info about the data
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 594194 entries, 0 to 594193
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   gender            594194 non-null  category
 1   SeniorCitizen     594194 non-null  int8    
 2   Partner           594194 non-null  category
 3   Dependents        594194 non-null  category
 4   tenure            594194 non-null  int8    
 5   PhoneService      594194 non-null  category
 6   MultipleLines     594194 non-null  category
 7   InternetService   594194 non-null  category
 8   OnlineSecurity    594194 non-null  category
 9   OnlineBackup      594194 non-null  category
 10  DeviceProtection  594194 non-null  category
 11  TechSupport       594194 non-null  category
 12  StreamingTV       594194 non-null  category
 13  StreamingMovies   594194 non-null  category
 14  Contract          594194 non-null  category
 15  PaperlessBilling  594194 non-null  category
 16  Payment

In [11]:
# Verify if there are null values
train.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [12]:
# Check for dublicates
print(train.duplicated().sum())
train.drop_duplicates()

4237


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
id,,,,,,,,,,,,,,,,,,,,
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.09375,1654.0000,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50000,3778.0000,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.37500,5840.0000,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.68750,70.6875,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.43750,70.4375,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.56250,5460.0000,No
594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.93750,6784.0000,No
594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40625,1872.0000,No


In [ ]:
# Set the columns names and select 'category'

names = train.columns
names

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [14]:
# Check for outliers

def remove_outliers(df):

    outliers_columns = []
    total_rows = len(df)
    
    for column in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = (df[column] < lower_bound) | (df[column] > upper_bound)
        outlier_count = outliers.sum()
        outlier_percentage = (outlier_count / total_rows) * 100
        
        if outlier_percentage >= 20:
            outliers_columns.append(column)
            print(f"Outliers detected in column '{column}': {outlier_percentage:.2f}% of total rows.")
            
            df = df[~outliers]
            print(f"Removed {outlier_count} outliers from column '{column}'.")
    
    if not outliers_columns:
        print("No columns with outliers exceeding 20% detected.")

    return df
    
train = remove_outliers(train)

No columns with outliers exceeding 20% detected.


In [ ]:
# One Hot Encode the training data
train = pd.get_dummies(train, columns = ['Gender', 'Vehicle_Age', 'Vehicle_Damage'])

In [ ]:
# Rename columns encoded
train.rename(columns = {'Vehicle_Age_< 1 Year': 'Vehicle Age Less than 1 yr',
                           'Vehicle_Age_1-2 Year': 'Vehicle Age between 1 and 2 yrs',
                           'Vehicle_Age_> 2 Years': 'Vehicle Age greater than 2 yrs'
                          }, inplace = True)

In [ ]:
# Set the dataframes from training data
response = train.pop('Response')
train.insert(len(train.columns), 'Response', response)
y_data = pd.DataFrame(train['Response'])
X_data = pd.DataFrame(train.iloc[:,:-1])

In [ ]:
# Reshape for unbalanced data
smoteenn = SMOTEENN()
X_data, y_data = smoteenn.fit_resample(X_data, y_data)

In [ ]:
# Split The Training And Testing Data

rows = train.shape[0]
cols = train.shape[1]

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

In [ ]:
# Fill the null values with zero

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [ ]:
# Newer input and output data
train

# Plotting Training Data

In [ ]:
# Plotting scatter training data for relevancy 

def scatter():
    numerical_columns = train.select_dtypes(include=['int8', 'float32', 'uint8', 'int16']).columns
    for column in numerical_columns[:-1]:
        plt.figure()
        plt.scatter(train[train['Response'] == 0][column], train[train['Response'] == 0]['Response'], label='No', color='red', alpha=0.7)
        plt.scatter(train[train['Response'] == 1][column], train[train['Response'] == 1]['Response'], label='Yes', color='blue', alpha=0.7)
        plt.legend()
        plt.title(column)
        plt.ylabel('Response')
        plt.xlabel(column)
        plt.grid()
        plot_path = f"{_plots}{column}_scatter.jpg"
        plt.savefig(plot_path)
        plt.show()

scatter()

In [ ]:
# Plotting histogram training data for relevancy 

def histogram():
    numerical_columns = train.select_dtypes(include=['int8', 'float32', 'uint8', 'int16']).columns
    for column in numerical_columns[:-1]:
        plt.figure()
        plt.hist(train[train['Response'] == 0][column], label='No', color='red', alpha=0.7, density=False)
        plt.hist(train[train['Response'] == 1][column], label='Yes', color='blue', alpha=0.7, density=False)
        plt.legend()
        plt.title(column)
        plt.ylabel(column)
        plt.xlabel('Response')
        plt.grid()
        plot_path = f"{_plots}{column}_histogram.jpg"
        plt.savefig(plot_path)  
        plt.show()

histogram()

# K-NearestNeighboor Model 

In [18]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn_model = KNeighborsClassifier()
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)
print(classification_report(y_test, knn_pred))

In [ ]:
pkl.dump(knn_model, open('./tested/KNearestNeighboor.pkl', 'wb'))
print(accuracy_score(y_test, knn_pred))
del knn_model, knn_pred